# 02 · From TF-IDF to InLegalBERT and sentence selection
**Question:** What did each model add, and how was it trained?

The path is **TF-IDF baselines → word BiLSTM → context-aware InLegalBERT → document BiLSTM (with a CRF comparison) → sentence selector**.
The final app uses the existing verified encoder and heads. Heavy retraining is off by default. Short cells expose the decisions; reusable loops are in `src/train_models.py` and `src/train_selector.py`. Original experiment sources remain in `research/`.

For visible, step-by-step KNN, logistic regression, SVM and word-BiLSTM training code, open the optional [Stage 1 notebook](00_classical_ml_and_word_bilstm.ipynb). [Where the rest of the code lives](../docs/code_map.md).

## Setup — run once
The project uses an isolated Python environment so its Hugging Face packages do not conflict with Colab's installed Gradio/Diffusers.
`%%project` means “run this ordinary Python cell in that environment.” Variables persist between these cells.
The first setup downloads dependencies and can take several minutes. No token is needed for the demo or downloaded trained model.
For the gated role dataset, accept its access conditions and add `HF_TOKEN` in Colab Secrets; never paste it into a cell.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path(os.environ.get('PROJECT_ROOT', '/content/legal-judgment-summarizer' if Path('/content').exists() else Path.cwd()))
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/Mananfrfr/legal-judgment-summarizer.git', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
from notebook_runtime import setup
setup(ROOT)


In [ ]:
%%project
from pathlib import Path
import os, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path(os.environ['PROJECT_ROOT'])


In [ ]:
%%project
from data_utils import demo_role_data, read_frames
from train_models import train_baselines
from datetime import datetime, timezone
DEMO = True
RUN_TRAINING = False
frames = demo_role_data()[0] if DEMO else read_frames(ROOT / 'data/processed/official')
run = ROOT / 'outputs' / datetime.now(timezone.utc).strftime('training_%Y%m%d_%H%M%S')
train, dev = frames['train'], frames['dev']


## 1. Classical machine learning
TF-IDF converts a sentence into a sparse vector of word importance. KNN finds nearby examples; logistic regression and SVM learn decision boundaries. The majority classifier is the minimum baseline. Fit the vocabulary and IDF weights on training text only.
The demo runs quickly and illustrates `.fit()` and `.predict()`; its performance is **not** a real evaluation result.

In [ ]:
%%project
baseline_scores = train_baselines(train, dev, run / 'baselines')
print('SYNTHETIC DEMO SCORES' if DEMO else 'NEW OFFICIAL DEV RUN')
display(baseline_scores)


## 2. Why a BiLSTM, and why then BERT?
The Stage 1 BiLSTM reads the words of one sentence in both directions. It learns order, but starts with randomly initialized word embeddings.
InLegalBERT starts with a language model pretrained on Indian legal text. Fine-tuning teaches it our 13 roles. Stage 2.2 compared **target-only 256 tokens** with **target plus neighboring sentences within 256 tokens**. The target gets at most 192 tokens in the context experiment. Neighbors must come from the same judgment.

In [ ]:
%%project
from stage44_model import SentenceBiLSTM, SentenceSelector
print('Document BiLSTM:', SentenceBiLSTM())
print('Sentence selector without role features:', SentenceSelector(with_roles=False))


## 3. Fine-tune the encoder (optional GPU run)
For each batch: predict roles → calculate cross-entropy loss → backpropagate → update weights. Four batches accumulate gradients before an optimizer step. We save the epoch with the best **development macro F1**, not the highest training accuracy.
Three epochs, learning rate 2e-5, effective batch size 32. The two experiments use the same data and principal settings.
These cells run only when `DEMO=False` and `RUN_TRAINING=True`; they can take hours. New outputs are separate from the deployed model.

In [ ]:
%%project
from train_models import train_encoder
if RUN_TRAINING and not DEMO:
    target_checkpoint = train_encoder(train, dev, run / 'target256', mode='target256')
    context_checkpoint = train_encoder(train, dev, run / 'context256', mode='context256')
else:
    print('Encoder training skipped. Recorded results are in Notebook 03.')


## 4. Read the whole judgment as a sequence
The second BiLSTM is different from the Stage 1 word model: it reads **768-dimensional sentence vectors**, in judgment order. It predicts one role per sentence. Padding is masked from the loss.
The optional CRF learns which role sequences are plausible and uses Viterbi decoding. It was a controlled experiment, not a guaranteed improvement. The deployed role head uses softmax.

In [ ]:
%%project
from train_models import document_embeddings, train_role_head
if RUN_TRAINING and not DEMO:
    train_sequences = document_embeddings(train, context_checkpoint)
    dev_sequences = document_embeddings(dev, context_checkpoint)
    for use_crf in [False, True]:
        train_role_head(train_sequences, dev_sequences, run / ('crf' if use_crf else 'softmax'),
                        context_checkpoint, use_crf=use_crf)
else:
    print('Document-head training skipped.')


## 5. How does the model learn what to include?
A role label is not an importance label. For IN-Abs training judgments, we create **weak sentence labels** using the training reference summary. One teacher ranks TF-IDF similarity; another greedily adds new reference unigram/bigram coverage. Both respect a 400-word budget.
These are heuristic labels, not expert sentence annotations. At inference the reference summary is unavailable and is never a model input.

In [ ]:
%%project
from selection_training import tfidf_budgeted_labels, greedy_oracle_labels
sentences = ['The claimant challenged the payment rejection before the court.',
             'The court found the claim was filed within time.',
             'The appeal is allowed and payment is directed within six weeks.']
training_reference = 'The appeal is allowed. Payment must be made within six weeks.'
display(pd.DataFrame({'sentence': sentences,
    'TF-IDF teacher': tfidf_budgeted_labels(sentences, training_reference, budget=24),
    'Coverage teacher': greedy_oracle_labels(sentences, training_reference, budget=24)}))


## 6. Train four matched selectors (optional)
Two teachers × role features on/off gives four conditions. Every selector uses a sentence embedding plus normalized position; the role-aware version also receives 13 predicted role probabilities. A small neural network produces an importance score. Binary cross-entropy trains it; development ROUGE-L selects the checkpoint.
This stage deliberately uses the original frozen encoder/head through `load_deployed_model`. Newly retrained encoders/heads require a new feature pass and provenance bundle; they must not be mixed with old selectors. New research checkpoints do not silently replace the app's verified checkpoints.

In [ ]:
%%project
from project_api import load_deployed_model
from train_selector import prepare_selector_cases, train_selectors
if RUN_TRAINING and not DEMO:
    frozen_model = load_deployed_model(ROOT)
    cases = prepare_selector_cases(ROOT / 'data/raw/dataset.zip', ROOT / 'data/splits.json', frozen_model)
    history = train_selectors(cases, run / 'selectors')
else:
    print('Selector training skipped. No weights changed during this demonstration.')


### Explain it to your faculty
“The encoder represents legal sentences; the document BiLSTM predicts their roles; a separately trained selector ranks importance. I compared role-aware selection with a no-role control instead of assuming roles would help.”

Optional CRF, weighted-loss experiments and the earlier word-level BiLSTM are documented in the source archive and methodology. Consolidated training entrypoints retain the main architectures/settings but have not been rerun to reproduce all recorded numbers. Notebook 03 labels the recorded runs explicitly.